# EmbdGuard: Real-Time Embedding Poisoning Detection for Recommender Systems

**GitHub:** [github.com/aliafzal/embdguard](https://github.com/aliafzal/embdguard)

---

## The Problem: Embedding Poisoning

Modern recommender systems represent users and items as learned embedding vectors. A Two-Tower model, for example, learns a user embedding and an item embedding, then scores candidates by dot product similarity. These embeddings are the core of the model — and they are vulnerable.

In a **data poisoning attack**, an adversary creates fake user accounts with carefully crafted interaction histories. When the model retrains on the poisoned data, the attacker's fake interactions shift the target item's embedding closer to many real users, causing the system to recommend that item more broadly. This is the basis of attacks like **DLAttack** (Huang et al., NDSS 2021), which uses gradient-based optimization on a surrogate model to generate maximally effective fake profiles.

The challenge: these attacks operate at the embedding level, not the application level. Traditional fraud detection (duplicate accounts, unusual click patterns) can miss sophisticated poisoning that produces statistically plausible interaction histories.

## What is EmbdGuard?

**EmbdGuard** is an embedding-level monitoring and defense framework for PyTorch recommender systems. It works by attaching hooks directly to the model's embedding layers, capturing fine-grained statistics at every training step, and running pluggable anomaly detectors to catch poisoning in real time.

### How It Works

EmbdGuard attaches two types of PyTorch hooks to each embedding table:

1. **Forward pre-hook** — captures which embedding rows (user/item IDs) are accessed in each batch, tracking access frequency and temporal patterns
2. **Backward hook** — captures gradient statistics (norm, max, kurtosis, concentration ratio) flowing back into each embedding table

These raw signals feed into a `StatAccumulator` (rolling window buffer), which detectors then analyze for anomalies.

### Detectors

EmbdGuard ships with 5 pluggable detectors, each targeting a different poisoning signal:

| Detector | Signal | What It Catches |
|----------|--------|-----------------|
| **GradientAnomalyDetector** | Z-score of gradient norm | Sudden gradient spikes from poisoned batches |
| **AccessFrequencyDetector** | Max/mean access count ratio | One item accessed disproportionately often |
| **EmbeddingDriftDetector** | Per-row L2 drift from reference | Slow-and-steady weight shifts on target items |
| **GradientDistributionDetector** | Gradient kurtosis + concentration | Gradient energy concentrated in few rows |
| **TemporalAccessDetector** | Jaccard overlap + burst score | Same item appearing in top-K every step |

### Defenses

When detectors fire alerts, EmbdGuard can activate defenses that modify gradient flow in real time:

| Defense | Mechanism |
|---------|-----------|
| **GradientClipDefense** | Clips per-row gradients on flagged rows to a max norm |
| **RowFreezeDefense** | Zeros gradients on flagged rows entirely |
| **InteractionFilterDefense** | Removes flagged items from the batch before forward pass |

All defenses auto-expire after a configurable duration to avoid permanently blocking legitimate updates.

### Architecture

```
Training Loop
  |
  |-- model.forward(batch)
  |     |-- EmbeddingBagCollection
  |           |-- [Forward hook] -> captured accessed row IDs
  |
  |-- loss.backward()
  |     |-- [Backward hook] -> captured gradient norms, kurtosis, concentration
  |
  |-- guard.step()
        |-- StatAccumulator: stores rolling window of stats per table
        |-- Detectors: analyze stats, emit Alert objects
        |-- Defenses: activate gradient clipping/freezing on flagged rows
```

### Usage

```python
from src.guard import EmbdGuard
from src.detectors.access_frequency import AccessFrequencyDetector
from src.defenses.row_freeze import RowFreezeDefense

guard = EmbdGuard(model)
guard.add_detector(AccessFrequencyDetector(concentration_threshold=5.0))
guard.add_defense(RowFreezeDefense())

for batch in dataloader:
    loss = model(batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    alerts = guard.step()  # monitor, detect, defend
```

## What This Notebook Covers

1. **Synthetic Demo** — EmbdGuard catching a poisoning attack in <10 training steps
2. **Detector Comparison** — precision, recall, F1, and latency for each detector
3. **Defense Validation** — verifying GradientClip, RowFreeze, and InteractionFilter
4. **MovieLens-1M Evaluation** — full DLAttack pipeline with EmbdGuard monitoring
5. **Results Analysis** — alert distributions, model quality, embedding drift
6. **Key Findings** — the scale gap between synthetic and real-world detection

---
## Setup

Clone the [EmbdGuard repository](https://github.com/aliafzal/embdguard) and install dependencies. The model uses a Two-Tower architecture with `EmbeddingBagCollection` for sparse features — when TorchRec is unavailable, pure-PyTorch fallbacks are used automatically.

In [ ]:
%%bash
# Clone repo (skip if already present)
if [ ! -d "embdguard" ]; then
    git clone https://github.com/aliafzal/embdguard.git
fi

# Install dependencies
pip install -q scikit-learn

# GPU diagnostics
echo "=== GPU diagnostics ==="
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader 2>/dev/null || echo "nvidia-smi not found"
python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA built: {torch.version.cuda}, Available: {torch.cuda.is_available()}')" 2>/dev/null
python -c "import torch; print(f'CUDA devices: {torch.cuda.device_count()}')" 2>/dev/null

echo "Setup complete"

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

REPO_ROOT = os.path.abspath("embdguard")
DLATTACK_DIR = os.path.join(REPO_ROOT, "dlattack_research")
sys.path.insert(0, REPO_ROOT)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

# CUDA diagnostics
print(f"PyTorch {torch.__version__}")
print(f"CUDA built-in version: {torch.version.cuda}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA devices: {torch.cuda.device_count()}")
    device = torch.device("cuda")
    torch.cuda.init()
    _ = torch.zeros(1, device=device)
    print(f"CUDA memory allocated: {torch.cuda.memory_allocated()/1024**2:.1f} MiB")
else:
    device = torch.device("cpu")
    print("WARNING: CUDA not available — running on CPU!")

print(f"\nDevice: {device}")

In [ ]:
# Import EmbdGuard modules
from src.guard import EmbdGuard
from src.detectors.gradient_anomaly import GradientAnomalyDetector
from src.detectors.access_frequency import AccessFrequencyDetector
from src.detectors.embedding_drift import EmbeddingDriftDetector
from src.detectors.gradient_distribution import GradientDistributionDetector
from src.detectors.temporal_access import TemporalAccessDetector
from src.defenses.gradient_clip import GradientClipDefense
from src.defenses.row_freeze import RowFreezeDefense
from src.defenses.interaction_filter import InteractionFilterDefense
from src.evaluation.harness import EvalRun, DataConfig, AttackConfig
from src.evaluation.compare import compare, format_comparison

print("EmbdGuard modules loaded successfully")

In [ ]:
# Import DLAttack modules (handle src namespace collision)
import src as embdguard_src

sys.path.insert(0, DLATTACK_DIR)
del sys.modules["src"]
for key in list(sys.modules.keys()):
    if key.startswith("src."):
        del sys.modules[key]

import src.dataset as dl_dataset
import src.model as dl_model
import src.train as dl_train
import src.attack as dl_attack
import src.evaluate as dl_evaluate

sys.modules["src"] = embdguard_src
sys.path.remove(DLATTACK_DIR)

print("DLAttack modules loaded successfully")

---
## Part 1: Synthetic Demo — EmbdGuard Catches Poisoning in <10 Steps

To demonstrate EmbdGuard's core detection capability, we build a small Two-Tower recommender (1,000 users, 2,000 items) and simulate a targeted poisoning attack:

- **Phase 1 (Clean training, 25 steps):** Normal training with random user-item pairs. EmbdGuard should remain silent — no anomalies.
- **Phase 2 (Poisoned training, 25 steps):** 80% of each batch targets item 42 — simulating a DLAttack where fake users all interact with the same target item.

This creates a stark statistical shift: one item suddenly dominates access patterns, receives outsized gradients, and its embedding drifts far from its reference position. EmbdGuard's detectors are designed to catch exactly these signals.

We attach 4 detectors:
- **GradientAnomalyDetector** (z-score > 3.0) — flags gradient spikes
- **AccessFrequencyDetector** (concentration > 5x) — flags items accessed far above average
- **EmbeddingDriftDetector** (drift > 3 sigma) — flags embeddings shifting from baseline
- **TemporalAccessDetector** (burst > 80% of window) — flags items appearing in top-K every step

In [ ]:
N_USERS, N_ITEMS = 1000, 2000
TARGET_ITEM = 42
BATCH = 128

# Build model
ebc = dl_model.build_ebc(N_USERS, N_ITEMS, 32, device=device)
two_tower = dl_model.TwoTower(ebc, layer_sizes=[64, 32], device=device)
model = dl_model.TwoTowerTrainTask(two_tower)
optimizer = dl_model.make_optimizer(model)

# Attach EmbdGuard with 4 detectors
guard = EmbdGuard(model)
guard.add_detector(GradientAnomalyDetector(threshold_z=3.0, min_steps=20))
guard.add_detector(AccessFrequencyDetector(concentration_threshold=5.0, min_steps=10))
guard.add_detector(EmbeddingDriftDetector(drift_threshold_z=3.0, min_steps=10))
guard.add_detector(TemporalAccessDetector(burst_window=5, burst_threshold=0.8, min_steps=10))

print(f"Model: {N_USERS} users x {N_ITEMS} items, embed_dim=32")
print(f"Detectors: GradientAnomaly, AccessFrequency, EmbeddingDrift, TemporalAccess")

In [ ]:
clean_losses, poison_losses = [], []
all_alerts = []

# Phase 1: Clean training
print("Phase 1: Clean training (25 steps)")
for step in range(25):
    users = torch.randint(0, N_USERS, (BATCH,), device=device)
    items = torch.randint(0, N_ITEMS, (BATCH,), device=device)
    labels = torch.cat([torch.ones(BATCH // 2, device=device), torch.zeros(BATCH // 2, device=device)])
    kjt = dl_model.make_kjt(users, items)
    loss, _ = model(kjt, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    alerts = guard.step()
    clean_losses.append(loss.item())
    all_alerts.extend(alerts)
    if step % 10 == 0:
        print(f"  Step {guard.step_count}: loss={loss.item():.4f} — {len(alerts)} alerts")

# Phase 2: Poisoned training
print(f"\nPhase 2: Poisoned training targeting item {TARGET_ITEM} (25 steps)")
for step in range(25):
    users = torch.randint(0, N_USERS, (BATCH,), device=device)
    n_target = int(BATCH * 0.8)
    items = torch.cat([
        torch.full((n_target,), TARGET_ITEM, dtype=torch.long, device=device),
        torch.randint(0, N_ITEMS, (BATCH - n_target,), device=device),
    ])
    labels = torch.ones(BATCH, device=device)
    kjt = dl_model.make_kjt(users, items)
    loss, _ = model(kjt, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    alerts = guard.step()
    poison_losses.append(loss.item())
    all_alerts.extend(alerts)
    if alerts:
        for a in alerts[:2]:  # Show first 2 per step
            print(f"  [{a.detector}] Step {a.step}: {a.message}")

guard.detach()
print(f"\nTotal alerts: {len(all_alerts)}")
if all_alerts:
    print(f"First alert at step {all_alerts[0].step} ({all_alerts[0].detector})")

In [ ]:
# Visualize the detection
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

steps = list(range(1, 51))
axes[0].plot(steps[:25], clean_losses, "steelblue", linewidth=2, label="Clean")
axes[0].plot(steps[25:], poison_losses, "red", linewidth=2, label="Poisoned")
axes[0].axvline(x=25.5, color="gray", linestyle="--", alpha=0.5, label="Poison injected")
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_title("Training Loss", fontsize=13)

# Group alerts by detector
det_colors = {
    "access_frequency": "red",
    "embedding_drift": "purple",
    "temporal_access": "green",
    "gradient_anomaly": "orange",
}
for a in all_alerts:
    axes[1].axvline(a.step, color=det_colors.get(a.detector, "gray"), alpha=0.5, linewidth=2)

# Legend
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color=c, linewidth=2, label=d) for d, c in det_colors.items()
           if any(a.detector == d for a in all_alerts)]
axes[1].axvline(x=25.5, color="gray", linestyle="--", alpha=0.5)
axes[1].legend(handles=handles, fontsize=9, loc="upper left")
axes[1].set_xlabel("Step", fontsize=12)
axes[1].set_ylabel("Alerts", fontsize=12)
axes[1].set_title("Detection Timeline", fontsize=13)
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

print(f"\nDetection latency: {all_alerts[0].step - 25} steps after injection")

EmbdGuard detects the poisoning within a few steps of injection. The `TemporalAccessDetector` typically fires first (it spots the same item appearing in top-K every batch), followed by `AccessFrequencyDetector` (concentration ratio >5x) and `EmbeddingDriftDetector` (target item drifting from reference).

This demonstrates the core value proposition: **EmbdGuard can catch poisoning attacks within a handful of training steps**, long before the attack has meaningfully shifted the model's recommendations.

---
## Part 2: Detector Comparison on Synthetic Data

Which detector is best? EmbdGuard includes a built-in evaluation harness (`EvalRun`) that runs the same clean-then-attack protocol with each detector configuration independently, computing standard detection metrics:

- **Precision** — what fraction of alerts are true positives (fired during attack phase)?
- **Recall** — did the detector fire at all during the attack phase?
- **F1** — harmonic mean of precision and recall
- **Detection latency** — how many steps after the attack starts before the first alert?

In [ ]:
configs = [
    {"name": "gradient_anomaly", "detectors": [GradientAnomalyDetector(threshold_z=3.0, min_steps=20)]},
    {"name": "access_frequency", "detectors": [AccessFrequencyDetector(concentration_threshold=5.0, min_steps=10)]},
    {"name": "embedding_drift", "detectors": [EmbeddingDriftDetector(drift_threshold_z=3.0, min_steps=10)]},
    {"name": "temporal_access", "detectors": [TemporalAccessDetector(burst_window=5, burst_threshold=0.8, min_steps=10)]},
    {"name": "all_combined", "detectors": [
        AccessFrequencyDetector(concentration_threshold=5.0, min_steps=10),
        EmbeddingDriftDetector(drift_threshold_z=3.0, min_steps=10),
        TemporalAccessDetector(burst_window=5, burst_threshold=0.8, min_steps=10),
    ]},
]

results = compare(configs, data_config=DataConfig(), attack_config=AttackConfig())
print(format_comparison(results))

In [ ]:
names = [r["config"] for r in results]
precisions = [r["precision"] for r in results]
recalls = [r["recall"] for r in results]
f1s = [r["f1"] for r in results]
latencies = [r["detection_latency"] if r["detection_latency"] is not None else 0 for r in results]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
x = np.arange(len(names))
w = 0.35

axes[0].bar(x - w/2, precisions, w, label="Precision", color="steelblue")
axes[0].bar(x + w/2, recalls, w, label="Recall", color="coral")
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=35, ha="right", fontsize=9)
axes[0].set_ylabel("Score")
axes[0].set_title("Precision / Recall")
axes[0].legend()
axes[0].set_ylim(0, 1.15)

axes[1].bar(x, f1s, color="seagreen")
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=35, ha="right", fontsize=9)
axes[1].set_ylabel("F1")
axes[1].set_title("F1 Score")
axes[1].set_ylim(0, 1.15)

axes[2].bar(x, latencies, color="goldenrod")
axes[2].set_xticks(x)
axes[2].set_xticklabels(names, rotation=35, ha="right", fontsize=9)
axes[2].set_ylabel("Steps")
axes[2].set_title("Detection Latency (lower = faster)")

plt.suptitle("Detector Performance on Synthetic Attack", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

On synthetic data (80% poison ratio, 2K items), all detectors achieve near-perfect precision/recall. `TemporalAccessDetector` has the lowest latency (fires within 2-3 steps), making it the fastest first responder.

---
## Part 3: Defense Mechanisms

Detection alone isn't enough — EmbdGuard also provides **active defenses** that respond to alerts by modifying gradient flow in real time. These defenses use PyTorch tensor hooks, so they work without modifying the optimizer or training loop.

| Defense | Mechanism | Trade-off |
|---------|-----------|----------|
| **GradientClipDefense** | Clips gradients on flagged rows to `max_norm` | Allows some learning, reduces attack influence |
| **RowFreezeDefense** | Zeros gradients on flagged rows completely | No updates to suspicious rows, more aggressive |
| **InteractionFilterDefense** | Removes flagged items from batch before forward pass | Prevents poisoned data from reaching the model |

All defenses auto-expire after a configurable `duration` (in training steps) — this prevents a false positive from permanently blocking a legitimate item.

In [ ]:
# Validate GradientClipDefense
print("=== GradientClipDefense ===")
test_ebc = dl_model.build_ebc(50, 100, 16, device=device)
test_tt = dl_model.TwoTower(test_ebc, layer_sizes=[32, 16], device=device)
test_model = dl_model.TwoTowerTrainTask(test_tt)

clip_def = GradientClipDefense(max_norm=0.001)
clip_def.apply(test_model)
clip_def.activate("t_item_id", [42], duration=5)

kjt = dl_model.make_kjt(torch.tensor([0, 1], device=device), torch.tensor([42, 10], device=device))
loss, _ = test_model(kjt, torch.tensor([1.0, 0.0], device=device))
loss.backward()

weight = test_model.two_tower.ebc.embedding_bags["t_item_id"].weight
if weight.grad is not None:
    clipped = weight.grad[42].norm().item()
    normal = weight.grad[10].norm().item()
    print(f"  Row 42 (clipped) grad norm: {clipped:.6f} <= max_norm=0.001")
    print(f"  Row 10 (normal)  grad norm: {normal:.6f}")
    print(f"  Reduction: {normal/max(clipped, 1e-10):.0f}x")
clip_def.remove()

# Validate RowFreezeDefense
print("\n=== RowFreezeDefense ===")
test_ebc = dl_model.build_ebc(50, 100, 16, device=device)
test_tt = dl_model.TwoTower(test_ebc, layer_sizes=[32, 16], device=device)
test_model = dl_model.TwoTowerTrainTask(test_tt)

freeze_def = RowFreezeDefense()
freeze_def.apply(test_model)
freeze_def.activate("t_item_id", [42], duration=5)

kjt = dl_model.make_kjt(torch.tensor([0, 1], device=device), torch.tensor([42, 10], device=device))
loss, _ = test_model(kjt, torch.tensor([1.0, 0.0], device=device))
loss.backward()

weight = test_model.two_tower.ebc.embedding_bags["t_item_id"].weight
if weight.grad is not None:
    frozen = weight.grad[42].norm().item()
    print(f"  Row 42 (frozen) grad norm: {frozen:.10f} (effectively zero)")
freeze_def.remove()

# Validate InteractionFilterDefense
print("\n=== InteractionFilterDefense ===")
filter_def = InteractionFilterDefense()
filter_def.activate("t_item_id", [42, 99], duration=5)

users = torch.tensor([0, 1, 2, 3, 4])
items = torch.tensor([10, 42, 20, 99, 30])
labels = torch.tensor([1.0, 1.0, 0.0, 1.0, 0.0])
fu, fi, fl = filter_def.filter_batch(users, items, labels)
print(f"  Before: {len(users)} interactions, items={items.tolist()}")
print(f"  After:  {len(fu)} interactions, items={fi.tolist()}")
print(f"  Items 42 and 99 removed from batch")
filter_def.remove()

# Auto-expiration
print("\n=== Auto-Expiration ===")
freeze_def = RowFreezeDefense()
freeze_def.activate("t_item_id", [42], duration=3)
for i in range(4):
    active = 42 in freeze_def.active_rows.get("t_item_id", [])
    print(f"  Step {i}: row 42 active = {active}")
    freeze_def.step()
print("  Defense expired after 3 steps as configured")

---
## Part 4: MovieLens-1M — Full DLAttack Pipeline

Parts 1-3 used synthetic data with a high poison ratio (80%). Now we test on the real **MovieLens-1M** dataset (6,040 users, 3,706 items, ~1M interactions) to see how EmbdGuard performs at realistic scale.

We replicate the **DLAttack** (Huang et al., "Data Poisoning Attacks to Deep Learning Based Recommender Systems", NDSS 2021) — a state-of-the-art multi-round poisoning attack:

1. **Build surrogate:** Copy the target model's weights into a surrogate, retrain on current data
2. **Optimize fake users:** Use gradient descent on the surrogate to craft fake user profiles that maximize the target item's dot-product similarity with real users
3. **Inject:** Add fake user interactions to the training set
4. **Retrain:** Update the target model on the poisoned dataset
5. **Repeat** for multiple rounds, each building on the previous round's poisoned model

EmbdGuard monitors the retraining phase (step 4) with all 5 detectors. The key question: **can per-step embedding monitors detect realistic-scale poisoning?**

In [ ]:
# Load MovieLens-1M
os.chdir(DLATTACK_DIR)
dl_dataset.download_ml1m()
df, n_users, n_items, user_map, item_map = dl_dataset.load_ratings()
train_df, test_df = dl_dataset.split_data(df)

print(f"MovieLens-1M loaded:")
print(f"  Users: {n_users:,}")
print(f"  Items: {n_items:,}")
print(f"  Train: {len(train_df):,} interactions")
print(f"  Test:  {len(test_df):,} interactions")

In [ ]:
# Pick a mid-popularity target item
counts = train_df["item_id"].value_counts()
mid = counts[(counts > 20) & (counts < 100)].index.tolist()
TARGET = int(mid[0]) if mid else int(train_df["item_id"].mode()[0])

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(counts.values, bins=100, edgecolor="black", alpha=0.7, color="steelblue")
ax.axvline(counts[TARGET], color="red", linestyle="--", linewidth=2,
           label=f"Target item {TARGET} (n={counts[TARGET]})")
ax.set_xlabel("Number of interactions", fontsize=11)
ax.set_ylabel("Number of items", fontsize=11)
ax.set_title("MovieLens-1M: Item Popularity Distribution", fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Build or load baseline model
EMBED_DIM = 64
LAYER_SIZES = [128, 64]
LR = 0.001

ebc = dl_model.build_ebc(n_users, n_items, EMBED_DIM, device=device)
two_tower = dl_model.TwoTower(ebc, layer_sizes=LAYER_SIZES, device=device)
model = dl_model.TwoTowerTrainTask(two_tower)
model.to(device)

ckpt_dir = os.path.join(DLATTACK_DIR, "checkpoints")
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, "baseline.pt")

if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state, strict=False)
    model.to(device)
    print(f"Loaded baseline checkpoint")
else:
    print("No checkpoint found — training baseline from scratch (10 epochs)...")
    optimizer = dl_model.make_optimizer(model, lr=LR)
    dl_train.train(model, optimizer, train_df, n_items,
                   epochs=10, batch_size=2048, n_neg=4,
                   device=device, save_path=ckpt_path)

optimizer = dl_model.make_optimizer(model, lr=LR)
print(f"Model device: {next(model.parameters()).device}")

baseline = dl_evaluate.evaluate(
    model.two_tower, test_df, train_df, n_items, n_neg=99, k=10, device=device
)
print(f"Baseline: HR@10={baseline['HR@K']:.4f}, NDCG@10={baseline['NDCG@K']:.4f}")

### Run Multi-Round DLAttack with EmbdGuard Monitoring

We run 3 attack rounds with 50 fake users per round. Each round adds ~1,550 fake interactions (50 users x 31 items each) to the ~1M existing interactions — a poison ratio of just **0.16%**.

Detector thresholds are tuned higher than the synthetic defaults to account for MovieLens-1M's scale:
- **GradientAnomaly**: z > 5.0 (vs 3.0 synthetic) — larger batches create more gradient noise
- **AccessFrequency**: concentration > 3.0x (vs 5.0) — popular items naturally have high counts
- **EmbeddingDrift**: z > 8.0 (vs 3.0) — all embeddings drift significantly during normal training
- **GradientDistribution**: kurtosis z > 5.0, concentration > 50.0 — real gradient distributions are heavier-tailed
- **TemporalAccess**: burst threshold 1.0 over window of 10 — require perfect burst to avoid flagging popular items

In [ ]:
N_ROUNDS = 3
M_FAKE = 50
RETRAIN_EPOCHS = 5
BATCH_SIZE = 2048

poisoned_train = train_df.copy()
fake_uid_start = n_users

round_metrics = []
all_alerts_by_round = {}
all_alerts_by_detector = defaultdict(list)
loss_history = []
alerts_per_epoch = []

# Snapshot target embedding before attack
emb_before = model.two_tower.ebc.embedding_bags["t_item_id"].weight.data[TARGET].clone()

for r in range(1, N_ROUNDS + 1):
    print(f"\n{'='*60}")
    print(f"  ATTACK ROUND {r}/{N_ROUNDS} — target={TARGET}, m={M_FAKE}")
    print(f"{'='*60}")

    # 1. Build surrogate
    max_uid = int(poisoned_train["user_id"].max()) + 1
    surr = dl_attack._build_surrogate(model, max_uid, n_items, EMBED_DIM, LAYER_SIZES, device)
    surr_task = dl_model.TwoTowerTrainTask(surr)
    surr_opt = dl_model.make_optimizer(surr_task, lr=LR)
    print(f"  Retraining surrogate ({RETRAIN_EPOCHS} epochs)...")
    dl_train.train(surr_task, surr_opt, poisoned_train, n_items,
                   epochs=RETRAIN_EPOCHS, batch_size=BATCH_SIZE,
                   device=device, eval_fn=None, save_path=None)

    # 2. Generate fake users
    n_prev = (r - 1) * M_FAKE
    print(f"  Generating {M_FAKE} fake users...")
    fake_df = dl_attack.generate_fake_users(
        surr_task.two_tower, TARGET, n_users=n_users + n_prev,
        n_items=n_items, m=M_FAKE, n_filler=30, n_optim_steps=200,
        fake_user_id_start=fake_uid_start, device=device,
    )
    fake_uid_start += M_FAKE

    # 3. Inject
    poisoned_train = pd.concat([poisoned_train, fake_df]).reset_index(drop=True)
    poison_pct = len(fake_df) / len(poisoned_train) * 100
    print(f"  Poisoned: {len(poisoned_train):,} interactions (+{len(fake_df)}, {poison_pct:.2f}%)")

    # 4. Expand model
    max_uid = int(poisoned_train["user_id"].max()) + 1
    new_tt = dl_attack._build_plain_two_tower(max_uid, n_items, EMBED_DIM, LAYER_SIZES, device)
    dl_attack._copy_weights_to_plain(model, new_tt)
    new_tt.to(device)
    model = dl_model.TwoTowerTrainTask(new_tt)
    model.to(device)
    optimizer = dl_model.make_optimizer(model, lr=LR)

    # 5. Retrain with EmbdGuard
    print(f"  Retraining with EmbdGuard monitoring...")
    guard = EmbdGuard(model)
    guard.add_detector(GradientAnomalyDetector(threshold_z=5.0, min_steps=50))
    guard.add_detector(AccessFrequencyDetector(concentration_threshold=3.0, min_steps=30))
    guard.add_detector(EmbeddingDriftDetector(
        drift_threshold_z=8.0, min_steps=50, snapshot_interval=100))
    guard.add_detector(GradientDistributionDetector(
        kurtosis_z=5.0, concentration_threshold=50.0, min_steps=50))
    guard.add_detector(TemporalAccessDetector(
        burst_window=10, burst_threshold=1.0, top_k=3, min_steps=50))

    pos_u = torch.tensor(poisoned_train["user_id"].values, dtype=torch.long, device=device)
    pos_i = torch.tensor(poisoned_train["item_id"].values, dtype=torch.long, device=device)

    round_alerts = []
    step = 0
    for epoch in range(1, RETRAIN_EPOCHS + 1):
        model.train()
        users, items, labels = dl_train._negative_sample_tensors(
            pos_u, pos_i, n_items, 4, device
        )
        n_samples = len(users)
        total_loss = 0.0
        epoch_alerts = 0

        for start in range(0, n_samples, BATCH_SIZE):
            end = min(start + BATCH_SIZE, n_samples)
            kjt = dl_model.make_kjt(users[start:end], items[start:end])
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                loss, _ = model(kjt, labels[start:end])
                optimizer.zero_grad()
                loss.backward()
            optimizer.step()
            step += 1
            alerts = guard.step()
            round_alerts.extend(alerts)
            epoch_alerts += len(alerts)
            total_loss += loss.item() * (end - start)

        train_loss = total_loss / n_samples
        loss_history.append((r, epoch, train_loss))
        alerts_per_epoch.append((r, epoch, epoch_alerts))
        print(f"    Epoch {epoch} | loss={train_loss:.4f} | alerts={epoch_alerts:,}")

    guard.detach()

    all_alerts_by_round[r] = round_alerts
    for a in round_alerts:
        all_alerts_by_detector[a.detector].append(a)

    # Evaluate
    met = dl_evaluate.evaluate(model.two_tower, test_df, train_df, n_items, n_neg=99, k=10, device=device)
    thr = dl_evaluate.target_item_hit_ratio(
        model.two_tower, TARGET, test_df, train_df, n_items, n_neg=99, k=10, device=device)
    round_metrics.append((r, met["HR@K"], met["NDCG@K"], thr))
    print(f"  HR@10={met['HR@K']:.4f} | NDCG@10={met['NDCG@K']:.4f} | Target HR@10={thr:.4f}")

emb_after = model.two_tower.ebc.embedding_bags["t_item_id"].weight.data[TARGET].clone()

---
## Part 5: Results Analysis

Let's analyze what EmbdGuard detected during the DLAttack rounds — which detectors fired, how many alerts were generated, and whether the target item was flagged.

In [ ]:
# Alert summary table
print(f"{'Detector':<25} {'Alerts':>10} {'First Step':>12}")
print("-" * 50)
for det in sorted(all_alerts_by_detector.keys()):
    als = all_alerts_by_detector[det]
    print(f"{det:<25} {len(als):>10,} {min(a.step for a in als):>12}")
total = sum(len(a) for a in all_alerts_by_round.values())
print(f"{'TOTAL':<25} {total:>10,}")

In [ ]:
# Alert distribution
det_names = sorted(all_alerts_by_detector.keys())
det_counts = {d: len(all_alerts_by_detector[d]) for d in det_names}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].barh(det_names, [det_counts[d] for d in det_names], color="steelblue")
axes[0].set_xlabel("Alert count (log scale)")
axes[0].set_title("Alerts by Detector")
axes[0].set_xscale("log")

other = {d: c for d, c in det_counts.items() if d != "embedding_drift"}
axes[1].barh(list(other.keys()), list(other.values()), color="coral")
axes[1].set_xlabel("Alert count")
axes[1].set_title("Alerts (excl. embedding_drift)")

plt.tight_layout()
plt.show()

In [ ]:
# Model quality across rounds
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rounds = [m[0] for m in round_metrics]
hr = [m[1] for m in round_metrics]
ndcg = [m[2] for m in round_metrics]
thr = [m[3] for m in round_metrics]

axes[0].plot(rounds, hr, "o-", color="steelblue", linewidth=2, markersize=8)
axes[0].axhline(baseline["HR@K"], color="gray", linestyle="--", label="Baseline")
axes[0].set_xlabel("Attack Round")
axes[0].set_ylabel("HR@10")
axes[0].set_title("Overall Rec Quality")
axes[0].legend()

axes[1].plot(rounds, ndcg, "o-", color="coral", linewidth=2, markersize=8)
axes[1].axhline(baseline["NDCG@K"], color="gray", linestyle="--", label="Baseline")
axes[1].set_xlabel("Attack Round")
axes[1].set_ylabel("NDCG@10")
axes[1].set_title("Ranking Quality")
axes[1].legend()

axes[2].plot(rounds, thr, "o-", color="red", linewidth=2, markersize=8)
axes[2].set_xlabel("Attack Round")
axes[2].set_ylabel("Target HR@10")
axes[2].set_title(f"Target Item {TARGET} Promotion")
axes[2].set_ylim(-0.01, max(0.05, max(thr) * 1.2))

plt.suptitle("MovieLens-1M: Model Quality Over Attack Rounds", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Target item embedding drift vs random items
item_weights = model.two_tower.ebc.embedding_bags["t_item_id"].weight.data

# Reload baseline for comparison
ebc_ref = dl_model.build_ebc(n_users, n_items, EMBED_DIM, device=device)
tt_ref = dl_model.TwoTower(ebc_ref, layer_sizes=LAYER_SIZES, device=device)
m_ref = dl_model.TwoTowerTrainTask(tt_ref)
if os.path.exists(ckpt_path):
    m_ref.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False), strict=False)
weights_ref = m_ref.two_tower.ebc.embedding_bags["t_item_id"].weight.data

sample_ids = np.random.choice(n_items, min(500, n_items), replace=False)
drifts = [(item_weights[i] - weights_ref[i]).norm().item() for i in sample_ids]
target_drift = (item_weights[TARGET] - weights_ref[TARGET]).norm().item()
pct = np.mean([d <= target_drift for d in drifts]) * 100

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(drifts, bins=50, alpha=0.7, color="steelblue", edgecolor="black")
ax.axvline(target_drift, color="red", linestyle="--", linewidth=2,
           label=f"Target {TARGET} (drift={target_drift:.3f}, {pct:.0f}th pctl)")
ax.set_xlabel("L2 drift from baseline")
ax.set_ylabel("Count")
ax.set_title("Embedding Drift: Baseline vs Post-Attack")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Which items were flagged?
for det_name in ["access_frequency", "temporal_access", "embedding_drift"]:
    if det_name in all_alerts_by_detector:
        flagged = set()
        for a in all_alerts_by_detector[det_name]:
            row = a.details.get("hottest_row") or a.details.get("row_id")
            if row is not None:
                flagged.add(int(row))
        target_hit = TARGET in flagged
        status = "FLAGGED" if target_hit else "not flagged"
        print(f"{det_name}: {len(flagged)} unique rows flagged — target {TARGET} {status}")
        if det_name != "embedding_drift":
            print(f"  Flagged rows: {sorted(flagged)[:15]}")
        else:
            print(f"  Coverage: {len(flagged)/n_items*100:.1f}% of all items")
        print()

---
## Part 6: Key Findings

### Synthetic Data (Parts 1-2)
- All detectors achieve **near-perfect precision/recall** on synthetic attacks (80% poison ratio, 2K items)
- `TemporalAccessDetector` is the **fastest** first responder (2-3 step latency)
- Combined detectors achieve **F1 > 0.99**

### MovieLens-1M (Parts 4-5)
- **DLAttack is ineffective at this scale**: 50 fake users/round = ~1,550 interactions out of ~1M (0.16%). The poisoned signal is overwhelmed by legitimate data. Target HR@10 stays at 0.0 across all rounds
- **Per-step detectors face a scale gap**: Thresholds tuned for synthetic data don't transfer to real data. `EmbeddingDriftDetector` alone generates hundreds of thousands of alerts because normal training causes widespread drift across all embeddings
- **Popular items dominate signal**: `AccessFrequencyDetector` flags naturally popular items (blockbusters with thousands of interactions), not the mid-popularity attack target

### Defense Validation (Part 3)
- All 3 defenses work correctly at the mechanism level: `GradientClipDefense` reduces flagged row gradients by orders of magnitude, `RowFreezeDefense` zeroes them completely, `InteractionFilterDefense` removes poisoned items from batches
- Auto-expiration works as configured — defenses don't permanently block rows

### Research Implications
- **EmbdGuard's detection mechanisms are functionally correct** and effective in controlled scenarios where the poison-to-clean signal ratio is high
- **Real-world detection at scale** requires fundamentally different approaches: **cross-epoch differential analysis** (comparing stat distributions before/after suspected poisoning periods), **user-level profiling** (flagging new accounts with anomalous interaction patterns), or **ensemble scoring** (combining multiple weak signals that individually fall below threshold)
- The DLAttack itself needs a **much higher poisoning ratio** or a different attack strategy to succeed on MovieLens-1M — this is consistent with findings in the original paper, which used smaller datasets
- This represents a genuine and open research challenge: the **defense-at-scale problem** is fundamentally different from the **detection-in-principle problem**

### Next Steps (contributions welcome on [GitHub](https://github.com/aliafzal/embdguard))
- Cross-epoch differential detectors that compare stat distributions between training windows
- User-level anomaly detection (TIA detector) that flags suspicious new accounts
- Adaptive thresholds that auto-calibrate based on dataset statistics
- Evaluation on larger-scale attacks (more fake users, stronger optimization)

In [ ]:
# Final comparison table
comparison = pd.DataFrame([
    {"Condition": "Baseline (no attack)",
     "HR@10": f"{baseline['HR@K']:.4f}",
     "NDCG@10": f"{baseline['NDCG@K']:.4f}",
     "Target HR@10": "0.0000",
     "Total Alerts": 0},
] + [
    {"Condition": f"After Round {m[0]}",
     "HR@10": f"{m[1]:.4f}",
     "NDCG@10": f"{m[2]:.4f}",
     "Target HR@10": f"{m[3]:.4f}",
     "Total Alerts": f"{sum(len(all_alerts_by_round.get(i, [])) for i in range(1, m[0]+1)):,}"}
    for m in round_metrics
])
print(comparison.to_string(index=False))